# Day 05 — Exception Handling + Retry Patterns + logging

========================================================
Module 01: Python + Async + FastAPI for LLM Engineering
Vidya Sankalp | Applied GenAI Engineering

WHAT THIS FILE COVERS:
  - try / except / finally / raise — the full exception flow
  - Catching specific exception types (not bare `except:`)
  - Custom exception classes for LLM-specific errors
  - Retry with exponential backoff (using the `logging` module, not print)
  - The logging module: getLogger, levels, basicConfig

WHY THIS MATTERS FOR LLM ENGINEERING:
  LLM APIs fail constantly in production:
    - Rate limits (HTTP 429) — retry after delay
    - Network timeouts — retry up to N times
    - Malformed JSON response — retry with a cleaner prompt
    - Invalid API key — fail immediately (no point retrying)
  Knowing WHICH exception to catch and WHAT to do is critical.


In [ ]:

import json
import logging
import time
import random
from typing import Any, Callable




## SECTION 1: THE logging MODULE

═════════════════════════════════════════════════════════════════════════════
Why logging instead of print()?
- print() always runs and always goes to stdout — no filtering
- logging lets you set LEVELS: DEBUG < INFO < WARNING < ERROR < CRITICAL
- In production, you set level=INFO so DEBUG messages are suppressed
- Log messages include timestamps, file names, line numbers automatically
- CloudWatch, Splunk, Datadog all ingest Python log output natively
This is EXACTLY the same pattern used in every file of the final project:
log = logging.getLogger(__name__)


In [ ]:

# Configure logging for this module.
# basicConfig should only be called ONCE in your entire application.
# Typically done in main.py or the entry point, not in every module.
# We call it here so the demo at the bottom of the file works standalone.
logging.basicConfig(
    level  = logging.INFO,                              # show INFO and above
    format = "%(asctime)s %(levelname)s [%(name)s] %(message)s",
    # Format breakdown:
    #   %(asctime)s   — timestamp: "2026-05-03 14:22:01,234"
    #   %(levelname)s — level:     "INFO", "WARNING", "ERROR"
    #   %(name)s      — logger:    module name (from getLogger(__name__))
    #   %(message)s   — your message
)

# __name__ is the module name (e.g. "day05_exception_handling_logging")
# Each module gets its own logger — this lets you filter logs per module
log = logging.getLogger(__name__)




## SECTION 2: CUSTOM EXCEPTION CLASSES

═════════════════════════════════════════════════════════════════════════════
Custom exceptions make your code self-documenting and easier to debug.
When a caller catches `LLMRateLimitError`, they know exactly what happened.
When a caller catches `Exception`, they know almost nothing.


In [ ]:

class LLMError(Exception):
    """
    Base exception for all LLM-related errors.

    All specific LLM exceptions inherit from this base class.
    This lets callers catch all LLM errors with one `except LLMError:` block,
    or catch specific ones with `except LLMRateLimitError:`.
    """
    pass



In [ ]:
class LLMRateLimitError(LLMError):
    """
    Raised when the LLM API returns HTTP 429 (Too Many Requests).

    This is a RETRYABLE error — wait and try again.
    The `retry_after` attribute tells the caller how long to wait.
    """
    def __init__(self, message: str, retry_after: float = 1.0):
        super().__init__(message)           # call parent Exception.__init__
        self.retry_after = retry_after      # seconds to wait before retrying



In [ ]:
class LLMAuthenticationError(LLMError):
    """
    Raised when the API key is invalid or missing.

    This is NOT retryable — fix the API key first.
    """
    pass



In [ ]:
class LLMJSONParseError(LLMError):
    """
    Raised when the LLM returns a response that is not valid JSON.

    This is SOMETIMES retryable with a cleaner prompt.
    """
    def __init__(self, message: str, raw_response: str = ""):
        super().__init__(message)
        self.raw_response = raw_response   # the raw string that failed to parse



In [ ]:
class LLMContextLengthError(LLMError):
    """
    Raised when the prompt exceeds the model's context window.

    This is NOT retryable — the prompt must be shortened first.
    """
    def __init__(self, message: str, prompt_tokens: int = 0, max_tokens: int = 0):
        super().__init__(message)
        self.prompt_tokens = prompt_tokens
        self.max_tokens    = max_tokens




## SECTION 3: EXCEPTION HANDLING PATTERNS

═════════════════════════════════════════════════════════════════════════════


In [ ]:

def parse_customer_csv_row(row: dict) -> dict:
    """
    Parse and validate a single row from customers.csv.

    Demonstrates:
    - try/except for specific exception types
    - Logging at appropriate levels
    - Returning a safe default vs re-raising

    Args:
        row: A raw dict from csv.DictReader.

    Returns:
        A cleaned and typed customer dict.
    """

    try:
        return {
            "customer_id": int(row["customer_id"]),       # str → int
            "name"       : f"{row['first_name']} {row['last_name']}".strip(),
            "email"      : row["email"].lower().strip(),
            "city"       : row["city"],
            "created_at" : row["created_at"],
        }

    except KeyError as e:
        # KeyError: a required column is missing from the CSV row
        # This usually means the CSV file has a different schema than expected
        log.warning(f"Missing column in customer row: {e}. Row: {row}")
        return {}   # return empty dict — caller should check for this

    except ValueError as e:
        # ValueError: a value cannot be converted to the expected type
        # Example: int("abc") raises ValueError
        log.warning(f"Type conversion failed for customer row: {e}. Row: {row}")
        return {}



In [ ]:
def safe_parse_llm_json(raw_response: str, expected_fields: list[str]) -> dict:
    """
    Parse an LLM JSON response with full error handling.

    Demonstrates the complete try/except/else/finally flow.

    try     → run the risky code
    except  → handle specific errors
    else    → runs ONLY if no exception occurred (often omitted, but useful)
    finally → ALWAYS runs, even if an exception occurred (used for cleanup)

    Args:
        raw_response    : Raw string from the LLM.
        expected_fields : Fields we expect to find in the parsed result.

    Returns:
        Parsed dict, or partial dict with available fields.

    Raises:
        LLMJSONParseError: If the response cannot be parsed at all.
    """

    log.info(f"Parsing LLM response ({len(raw_response)} chars)...")

    # Track whether parsing succeeded — used in finally block
    parse_start = time.perf_counter()
    parsed: dict = {}

    try:
        # Strip markdown fences if present
        cleaned = raw_response.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("\n", 1)[-1].rsplit("```", 1)[0].strip()

        # Main parsing step — can raise json.JSONDecodeError
        parsed = json.loads(cleaned)
        log.debug(f"JSON parsed successfully: {list(parsed.keys())}")

    except json.JSONDecodeError as e:
        # Wrap the standard library error in our custom error class
        # This gives callers more context (the raw_response that failed)
        log.error(f"Failed to parse LLM JSON response: {e}")
        raise LLMJSONParseError(
            f"LLM returned invalid JSON: {e}",
            raw_response=raw_response,
        )

    else:
        # `else` block runs only if NO exception was raised in `try`
        # Here we do post-parse validation
        missing = [f for f in expected_fields if f not in parsed]
        if missing:
            log.warning(f"LLM response missing expected fields: {missing}")

    finally:
        # `finally` block ALWAYS runs — even if an exception was raised
        # Use it for cleanup: closing files, recording metrics, etc.
        elapsed_ms = round((time.perf_counter() - parse_start) * 1000, 2)
        log.info(f"Parse attempt finished in {elapsed_ms}ms")

    return parsed




## SECTION 4: RETRY WITH EXPONENTIAL BACKOFF + LOGGING

═════════════════════════════════════════════════════════════════════════════


In [ ]:

def call_llm_api_with_retry(
    prompt: str,
    model: str = "gpt-4o",
    max_retries: int = 3,
) -> str:
    """
    Call an LLM API with intelligent retry logic.

    Different errors require different responses:
    - Rate limit (429) → wait the requested time, then retry
    - Timeout         → short wait, then retry
    - Auth error      → fail immediately (retrying won't help)
    - JSON parse error → retry once with a cleaner prompt instruction

    Args:
        prompt     : The prompt to send to the LLM.
        model      : Which model to use.
        max_retries: Maximum number of retry attempts.

    Returns:
        The LLM response string.

    Raises:
        LLMAuthenticationError: Immediately, without retrying.
        LLMError: After all retries are exhausted.
    """

    attempt = 0
    last_error: Exception | None = None

    while attempt < max_retries:
        try:
            log.info(f"LLM call attempt {attempt + 1}/{max_retries} | model={model}")

            # Simulate different error scenarios (in real code this is httpx/openai call)
            response = _simulate_llm_call(prompt, attempt)

            log.info(f"LLM call succeeded on attempt {attempt + 1}")
            return response

        except LLMAuthenticationError as e:
            # Authentication errors are NOT retryable — fail immediately
            # re-raise without attempting again
            log.error(f"Authentication failed — check your API key: {e}")
            raise   # re-raise the same exception (keeps original traceback)

        except LLMRateLimitError as e:
            # Rate limit — wait the server-specified time before retrying
            wait = e.retry_after
            log.warning(f"Rate limited. Waiting {wait}s before retry...")
            time.sleep(wait)
            last_error = e

        except LLMJSONParseError as e:
            # JSON error — add instruction to the prompt and retry
            log.warning(f"LLM returned invalid JSON. Retrying with stricter instruction.")
            prompt = prompt + "\n\nIMPORTANT: Respond ONLY with valid JSON. No other text."
            last_error = e

        except TimeoutError as e:
            # Timeout — brief wait then retry
            wait = 2 ** attempt   # exponential: 1, 2, 4 seconds
            log.warning(f"Timeout on attempt {attempt + 1}. Waiting {wait}s...")
            time.sleep(wait)
            last_error = e

        attempt += 1

    # All retries exhausted
    log.error(f"All {max_retries} attempts failed. Last error: {last_error}")
    raise LLMError(f"LLM call failed after {max_retries} attempts") from last_error



In [ ]:
def _simulate_llm_call(prompt: str, attempt: int) -> str:
    """
    Simulate an LLM API call for demonstration purposes.

    Attempt 0 → rate limit error
    Attempt 1 → timeout
    Attempt 2 → success
    """
    if attempt == 0:
        raise LLMRateLimitError("429 Too Many Requests", retry_after=0.5)
    elif attempt == 1:
        raise TimeoutError("Request timed out after 30s")
    else:
        return '{"category": "TRACK_ORDER", "confidence": "high"}'




In [ ]:
if __name__ == "__main__":
    print("=" * 60)
    print("DAY 05 — Exception Handling + logging + Retry")
    print("=" * 60)

    # Section 1: Custom exceptions
    print("\n[1] Custom Exception Classes")
    try:
        raise LLMRateLimitError("Too many requests", retry_after=2.5)
    except LLMRateLimitError as e:
        print(f"  Caught LLMRateLimitError: {e}")
        print(f"  retry_after: {e.retry_after}s")
    except LLMError as e:
        # This would catch any other LLM error (but not reach here)
        print(f"  Caught generic LLMError: {e}")

    # Section 2: safe JSON parsing
    print("\n[2] Safe JSON Parsing")
    test_cases = [
        ('{"category": "URGENT", "confidence": "high"}', ["category", "confidence", "reason"]),
        ('```json\n{"intent": "TRACK_ORDER"}\n```', ["intent"]),
        ('This is not JSON at all', ["field"]),
    ]
    for raw, fields in test_cases:
        try:
            result = safe_parse_llm_json(raw, fields)
            print(f"  Parsed OK: {result}")
        except LLMJSONParseError as e:
            print(f"  Parse failed (expected): {e}")

    # Section 3: retry with logging
    print("\n[3] Retry with Exponential Backoff")
    try:
        result = call_llm_api_with_retry(
            prompt="Route this query: Where is my order?",
            max_retries=4,
        )
        print(f"  Final result: {result}")
    except LLMError as e:
        print(f"  Failed: {e}")

    # Section 4: CSV parsing with error handling
    print("\n[4] CSV Row Parsing with Error Handling")
    good_row  = {"customer_id": "1001", "first_name": "Danielle", "last_name": "Johnson", "email": "john21@example.net", "city": "Port Matthew", "created_at": "2023-11-24"}
    bad_row   = {"customer_id": "abc",  "first_name": "Test",     "last_name": "User",    "email": "test@example.com", "city": "NYC", "created_at": "2023-01-01"}
    empty_row = {}

    for row in [good_row, bad_row, empty_row]:
        result = parse_customer_csv_row(row)
        if result:
            print(f"  Parsed: {result['name']} (id={result['customer_id']})")
        else:
            print(f"  Failed to parse row (returned empty dict)")


## Run the demo

Run the cell below to execute the `if __name__ == '__main__':` block.


In [ ]:
# Execute the demo for this day
import runpy
runpy.run_path('modules/day05_exception_handling_logging.py', run_name='__main__')


### LLM API Retry — Exponential Backoff

LLM APIs return `HTTP 429 (Too Many Requests)` when you call them too fast.
The production pattern: **wait and retry**, doubling the wait each time.

```
Attempt 1 fails → wait 0.1s
Attempt 2 fails → wait 0.2s
Attempt 3 fails → wait 0.4s
Attempt 4 fails → wait 0.8s
```

`while` is the right tool here — you don't know which attempt will succeed.


In [ ]:
import time

def call_llm_api(query):
    """Simulates an API call that fails twice then succeeds."""
    call_llm_api.count = getattr(call_llm_api, 'count', 0) + 1
    if call_llm_api.count < 3:
        raise Exception("HTTP 429: Rate limit exceeded")
    return f"[LLM] Order #3042 is In Transit, arriving Friday."

max_retries = 4
attempt     = 0
response    = None

while attempt < max_retries:
    try:
        response = call_llm_api("Where is order #3042?")
        print(f"  Attempt {attempt + 1}: SUCCESS")
        break                               # exit loop on success
    except Exception as e:
        wait = 0.1 * (2 ** attempt)         # 0.1s, 0.2s, 0.4s, 0.8s
        print(f"  Attempt {attempt + 1}: FAILED — {e}. Waiting {wait:.1f}s")
        time.sleep(wait)
        attempt += 1

if response:
    print(f"Response: {response}")
else:
    print(f"All {max_retries} attempts failed.")
